# Hard-Cap Your `openai-agents` Bill With Out-of-Process Budget Guardrails

Agent loops can run away.  A retry loop, a misbehaving tool, or a recursive
hand-off can burn dollars before any in-process kill switch fires.  In-process
budget tracking is best-effort: if the process dies the count is lost, and if
multiple agent pods share a budget they will silently double-spend.

This cookbook shows how to add a **hard, fail-closed budget cap** to an
`openai-agents` agent using [Agentic SpendGuard](https://github.com/m24927605/agentic-spendguard),
an out-of-process, cryptographically-audited budget ledger.

Two integration points are shown:

1. **In-process Guardrail.**  A `@input_guardrail` decorator calls SpendGuard
   before the LLM is invoked.  The Agents SDK runtime sees the deny natively as
   `InputGuardrailTripwireTriggered`.  This is what the notebook demonstrates
   end-to-end.
2. **`OPENAI_BASE_URL` passthrough.**  A zero-code alternative: point your
   `OPENAI_BASE_URL` at a SpendGuard egress proxy that speaks both
   `/v1/chat/completions` and `/v1/responses`.  Calls over the budget come
   back as a 402-shaped error before the LLM is ever called.  Sketched at the
   end of the notebook; useful when you cannot or do not want to add a
   guardrail.

> The notebook runs in **mock mode** by default.  A stub SpendGuard client
> stands in for the real SDK so the notebook executes top-to-bottom without
> needing a sidecar, a Postgres ledger, or an OpenAI key.  Flip
> `MOCK_MODE = False` and follow the prerequisites in the final cell to run
> against a live SpendGuard stack.

## 1. Install dependencies

In [1]:
%pip install --quiet openai-agents

# In production, also:
# %pip install --pre 'spendguard-sdk[agents]>=0.4'
# The notebook uses an inline mock so it can run offline.

Note: you may need to restart the kernel to use updated packages.


## 2. Configuration

Two knobs.  `MOCK_MODE = True` runs without an OpenAI key or a SpendGuard
sidecar.  `BUDGET_CAP_USD` is the hard cap we'll enforce.

In [2]:
import os

MOCK_MODE = True            # set False to run against a real SpendGuard stack
BUDGET_CAP_USD = 0.50       # hard cap for the demo
ESTIMATED_COST_PER_CALL_USD = 0.20   # what we project each LLM call to cost

## 3. The SpendGuard client

In production this is `from spendguard import SpendGuardClient`.  Here we
inline a mock that mimics the same surface — connect over UDS gRPC, hand-shake,
and call `request_decision(...)` before each LLM call to reserve budget
atomically against an out-of-process ledger.

The real client returns a protobuf `DecisionOutcome` with a `decision_id` that
maps to a signed audit-chain row.  The mock returns a small dataclass with the
same fields a reader cares about.

In [3]:
import uuid
from dataclasses import dataclass, field


@dataclass
class DecisionOutcome:
    decision_kind: str        # "ALLOW" | "DENY"
    decision_id: str
    reason: str = ""


@dataclass
class MockSpendGuardClient:
    """Stand-in for `spendguard.SpendGuardClient`.

    Mirrors the parts of the production surface that matter for the demo:
    a budget cap, atomic reservation against the cap, and a decision_id
    that would map to a signed audit-chain row in production.
    """

    budget_cap_usd: float
    spent_usd: float = 0.0
    calls: list[DecisionOutcome] = field(default_factory=list)

    async def handshake(self) -> None:
        # Real client opens a UDS gRPC channel + does mTLS handshake here.
        return None

    async def request_decision(self, *, projected_cost_usd: float) -> DecisionOutcome:
        decision_id = str(uuid.uuid4())
        if self.spent_usd + projected_cost_usd <= self.budget_cap_usd:
            self.spent_usd += projected_cost_usd
            outcome = DecisionOutcome("ALLOW", decision_id, "within budget")
        else:
            outcome = DecisionOutcome(
                "DENY", decision_id,
                f"BUDGET_EXHAUSTED: ${self.spent_usd:.2f} spent, "
                f"${projected_cost_usd:.2f} would exceed ${self.budget_cap_usd:.2f} cap",
            )
        self.calls.append(outcome)
        return outcome


sg = MockSpendGuardClient(budget_cap_usd=BUDGET_CAP_USD)
print(f"SpendGuard mock initialized — cap ${sg.budget_cap_usd:.2f}")

SpendGuard mock initialized — cap $0.50


## 4. Wire SpendGuard as an Agents SDK `@input_guardrail`

The `@input_guardrail` decorator from the Agents SDK is the cleanest hook:
the function runs **before** each LLM invocation, and returning
`tripwire_triggered=True` short-circuits the call entirely.  The agent
runtime surfaces the trip as `InputGuardrailTripwireTriggered`, which we
catch at the call site.

In [4]:
from agents import input_guardrail, GuardrailFunctionOutput


@input_guardrail(name="spendguard_budget")
async def spendguard_budget_guardrail(ctx, agent, input_data):
    """Reserve budget against the SpendGuard ledger before each LLM call.

    Returns a tripwire (deny) when the reservation fails, which the Agents SDK
    surfaces as InputGuardrailTripwireTriggered to the caller.
    """
    outcome = await sg.request_decision(projected_cost_usd=ESTIMATED_COST_PER_CALL_USD)
    return GuardrailFunctionOutput(
        output_info={"decision_id": outcome.decision_id, "reason": outcome.reason},
        tripwire_triggered=(outcome.decision_kind == "DENY"),
    )

## 5. Build the agent

A trivial agent that we'll call repeatedly to drive spend.  The guardrail is
attached via the `input_guardrails=` list — no other changes to the agent
itself.

In [5]:
from agents import Agent

agent = Agent(
    name="summary-agent",
    instructions="You are a concise summarizer. Respond in one sentence.",
    input_guardrails=[spendguard_budget_guardrail],
)
print(f"Agent {agent.name!r} ready with {len(agent.input_guardrails)} guardrail(s)")

Agent 'summary-agent' ready with 1 guardrail(s)


## 6. Path 1 — under budget

`BUDGET_CAP_USD = 0.50` and each call is projected at $0.20, so the first two
calls succeed.  In MOCK_MODE we skip the actual LLM round-trip and just verify
the guardrail allowed the call; in real mode this would be a `Runner.run(...)`
invocation.

In [6]:
import asyncio
from agents import InputGuardrailTripwireTriggered, Runner


async def try_one_turn(turn_idx: int) -> None:
    if MOCK_MODE:
        # Mock mode: invoke the guardrail directly so the demo can run
        # without an OpenAI key.  In real mode (below) Runner.run(...) does
        # this for us and raises InputGuardrailTripwireTriggered natively
        # when the guardrail returns tripwire_triggered=True.
        outcome = await sg.request_decision(projected_cost_usd=ESTIMATED_COST_PER_CALL_USD)
        if outcome.decision_kind == "DENY":
            print(f"  turn {turn_idx}: DENIED   reason={outcome.reason!r}")
        else:
            print(f"  turn {turn_idx}: ALLOWED  "
                  f"(decision_id={outcome.decision_id[:8]}..., "
                  f"spent ${sg.spent_usd:.2f} / ${sg.budget_cap_usd:.2f})")
    else:
        try:
            result = await Runner.run(agent, "Summarize: the OpenAI Agents SDK")
            print(f"  turn {turn_idx}: ALLOWED — {result.final_output[:60]}...")
        except InputGuardrailTripwireTriggered as exc:
            info = exc.guardrail_result.output.output_info
            print(f"  turn {turn_idx}: DENIED   reason={info['reason']!r}")


async def main_under_budget():
    print("=== Path 1: under budget ===")
    for i in range(2):
        await try_one_turn(i + 1)

await main_under_budget()

=== Path 1: under budget ===
  turn 1: ALLOWED  (decision_id=c64a89c5..., spent $0.20 / $0.50)
  turn 2: ALLOWED  (decision_id=0e4d6b33..., spent $0.40 / $0.50)


## 7. Path 2 — over budget

The third call would push us over the $0.50 cap.  The guardrail trips before
the LLM is invoked — the decision happens **out-of-process** in the ledger,
not in this Python process, so the budget invariant holds even if multiple
agent processes share the same `budget_id`.

In [7]:
async def main_over_budget():
    print("\n=== Path 2: over budget ===")
    for i in range(3):
        await try_one_turn(i + 3)
    print(f"\nFinal: {len(sg.calls)} total decisions, "
          f"${sg.spent_usd:.2f} reserved, "
          f"{sum(1 for c in sg.calls if c.decision_kind == 'DENY')} denied.")

await main_over_budget()


=== Path 2: over budget ===
  turn 3: DENIED   reason='BUDGET_EXHAUSTED: $0.40 spent, $0.20 would exceed $0.50 cap'
  turn 4: DENIED   reason='BUDGET_EXHAUSTED: $0.40 spent, $0.20 would exceed $0.50 cap'
  turn 5: DENIED   reason='BUDGET_EXHAUSTED: $0.40 spent, $0.20 would exceed $0.50 cap'

Final: 5 total decisions, $0.40 reserved, 3 denied.


## 8. The zero-code alternative: `OPENAI_BASE_URL` passthrough

If you do not want to add a guardrail at all, SpendGuard ships an egress proxy
that speaks both `/v1/chat/completions` and `/v1/responses`.  Point the SDK at
the proxy and budgets are enforced at the HTTP layer — calls over the cap
return a 402-shaped error before the LLM is invoked.

```bash
# In your shell:
export OPENAI_BASE_URL="http://localhost:8088/v1"   # SpendGuard egress proxy
export OPENAI_API_KEY="sk-..."                       # your real key, forwarded
```

```python
# No agent-side code change needed:
from agents import Agent, Runner
agent = Agent(name="summary-agent", instructions="...")
result = await Runner.run(agent, "Summarize: ...")
# If the budget is exhausted, the call fails with an OpenAI-shaped 402 error.
```

This works for `chat.completions`, `responses`, and SSE streaming — the proxy
parses the streamed usage frames and commits the reservation on the
`response.completed` event.  Pick this path when you cannot modify the agent
code; pick the guardrail path when you want the agent runtime to see the deny
as a structured signal.

## 9. Production setup

To run this notebook against a real SpendGuard stack:

| Step | What |
|---|---|
| 1. **Sidecar + ledger** | Clone <https://github.com/m24927605/agentic-spendguard> and run `make demo-up`.  Brings up sidecar + ledger + Postgres on Docker Compose. |
| 2. **Real SDK** | `pip install --pre 'spendguard-sdk[agents]>=0.4'` |
| 3. **Swap the client** | Replace the `MockSpendGuardClient` cell with `sg = SpendGuardClient(socket_path="/var/run/spendguard/adapter.sock", tenant_id=...); await sg.connect(); await sg.handshake()` |
| 4. **Flip the flag** | `MOCK_MODE = False`, set `OPENAI_API_KEY` |

The guardrail wiring (cell 4), agent definition (cell 5), and demo loops (cells 6–7) are unchanged between mock and real modes — that is the integration point.

## Further reading

- SpendGuard repo: <https://github.com/m24927605/agentic-spendguard>
- SpendGuard's openai-agents integration doc: <https://github.com/m24927605/agentic-spendguard/blob/main/docs/site/docs/integrations/openai-agents.md>
- Microsoft AGT integration (analogous pattern, accepted upstream): <https://github.com/microsoft/agent-governance-toolkit/pull/2398>
- `openai-agents` SDK Guardrails docs: <https://openai.github.io/openai-agents-python/guardrails/>